# Mahd — Cry Detection with a CRNN

The detection model decides **every second** whether the baby is crying. It uses the windows and log-mel spectrograms of
the Data Preparation notebook (Sections 7–10) and is evaluated once on the held-out test split.

## Data

| | Windows (1 s) | Cry |
|---|---:|---:|
| Train/Validation | 158 756 | 14.8 % |
| — used for training | **149 325** | |
| **Test** (5 932 recordings) | **54 002** | **5.9 %** |

## Model

| Item | Value |
|---|---|
| Input | mono 16 kHz audio, **1-s windows**, hop 1 s, full band |
| Log-mel | `n_fft` 1024, window 400 (25 ms), hop 160 (10 ms), 64 mel bands 50–8000 Hz, `center=False`, `10·log10(max(S, 1e-10))` → **64 × 94** |
| Loudness normalisation | `level_db = 20·log10(rms + 1e-10)`; `gain_db = −20 − max(level_db, −50)` added to the log-mel |
| Standardisation | `(x + 31.47) / 15.01` (mean and standard deviation of the training log-mels) |
| Context | the current second and the **4 previous seconds** (5 spectrograms); a missing earlier second is zeros with a flag |
| Network | per second: 4 blocks (16, 32, 64, 128 channels) of [Conv 3×3 – BatchNorm – ReLU] × 2 + average pooling 2×2 → mean over frequency → mean and max over time → dense 128; the 5 embeddings (+ flag) → **GRU (128 units)** from the oldest to the current second → dropout 0.3 → dense 1 (cry score). 426 193 weights, 1.72 MB |
| Training | all train/validation windows with clean labels; binary cross-entropy weighted by class and source; band-limited copy with p = 0.5; SpecAugment; AdamW (lr 2·10⁻³, weight decay 10⁻⁴), one-cycle schedule, 15 epochs, batch 256, mixed precision, seed 42 |
| Decision | score ≥ threshold, the threshold with the best F1 on the out-of-fold validation scores |
| Alert | **2 consecutive seconds** classified as cry |
| Files for the device | `models/final_detection/CRNN.pt` (weights, log-mel statistics) and `CRNN_deploy.json` (threshold and settings, Section 2) |

## Results on the test set

| Metric | CRNN |
|---|---:|
| Average precision (AP) | **0.940** (95 % CI 0.930 – 0.949) |
| F1 Cry / F1 Non-Cry / **Macro-F1** | 0.885 / 0.993 / **0.939** |
| Precision / Recall (Cry) | 0.905 / 0.867 |
| MCC | 0.879 |
| Cry episodes detected | 84.6 % |
| False alerts per hour | 2.98 |
| Mean alert delay | 2.20 s |
| CPU time per second of audio (1 thread) | 2.4 ms log-mel + 16.2 ms CRNN |

# 1. Integration code

Everything the device needs: the log-mel of each second, the loudness normalisation, the CRNN, and a `CryDetector` that
keeps the last 4 seconds as context, applies the threshold and raises **one alert per cry episode** after 2 consecutive
cry seconds. Call `reset()` when a new audio stream starts.

In [ ]:
# ===== Mahd Cry Detection — inference code for the device (CRNN, 1 s + 4 s context) =====
# Same signal processing and network as training (Data Preparation Sections 5, 7.1, 10.1; final training below).
import json, collections
import numpy as np, librosa, torch, torch.nn as nn, torch.nn.functional as F

SR = 16000                       # mono, 16 kHz, full band
WIN = SR                         # one decision per 1-s window, hop 1 s
CONTEXT_S = 4                    # the 4 previous seconds are used as context
MEL = dict(sr=SR, n_fft=1024, win_length=400, hop_length=160, n_mels=64, fmin=50, fmax=8000, center=False, power=2.0)
NORM_TARGET_DB, NORM_FLOOR_DB = -20.0, -50.0     # per-window loudness normalisation


class Encoder(nn.Module):
    def __init__(self, ch=(16, 32, 64, 128), emb=128):
        super().__init__(); layers, c_in = [], 1
        for c in ch:
            layers += [nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                       nn.Conv2d(c, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True), nn.AvgPool2d(2)]
            c_in = c
        self.conv = nn.Sequential(*layers); self.fc = nn.Linear(2 * c_in, emb)

    def forward(self, x):
        h = self.conv(x.unsqueeze(1)).mean(2)
        return F.relu(self.fc(torch.cat([h.mean(-1), h.amax(-1)], 1)))


class CRNN(nn.Module):
    def __init__(self):
        super().__init__(); self.enc = Encoder(); self.gru = nn.GRU(129, 128, batch_first=True)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(128, 1))

    def forward(self, x, ok):                    # x: (batch, 5, 64, 94), index 0 = current second; ok: (batch, 5)
        b, n = x.shape[:2]
        e = self.enc(x.flatten(0, 1)).view(b, n, -1) * ok[..., None]
        out, _ = self.gru(torch.cat([e, ok[..., None]], 2).flip(1).float())
        return self.head(out[:, -1])             # cry score (logit)


def logmel_1s(window):
    """64 x 94 log-mel (dB) of one 1-s window."""
    m = librosa.feature.melspectrogram(y=np.asarray(window, np.float32), **MEL)
    return 10 * np.log10(np.maximum(m, 1e-10))


def loudness_gain_db(window):
    level = 20 * np.log10(np.sqrt(np.mean(np.asarray(window, np.float64) ** 2)) + 1e-10)
    return NORM_TARGET_DB - max(level, NORM_FLOOR_DB)


class CryDetector:
    """Feed one second of audio at a time; returns the cry score, the per-second decision and the alert."""

    def __init__(self, model_path, deploy_path, device="cpu"):
        ck = torch.load(model_path, map_location=device)
        cfg = json.load(open(deploy_path))
        self.model = CRNN().to(device); self.model.load_state_dict(ck["state_dict"]); self.model.eval()
        self.mean, self.std = ck["logmel_stats"]
        self.threshold, self.alert_s, self.device = cfg["threshold"], cfg["alert_seconds"], device
        self.reset()

    def reset(self):
        """Call when a new stream starts (no context from an earlier stream)."""
        self.context = collections.deque(maxlen=CONTEXT_S + 1)       # newest first
        self.cry_run = 0

    @torch.no_grad()
    def step(self, window):
        assert len(window) == WIN, "exactly 1 s of 16-kHz audio"
        x = (logmel_1s(window) + loudness_gain_db(window) - self.mean) / self.std
        self.context.appendleft(x.astype(np.float32))
        X = np.zeros((1, CONTEXT_S + 1) + x.shape, np.float32); ok = np.zeros((1, CONTEXT_S + 1), np.float32)
        for i, m in enumerate(self.context):
            X[0, i], ok[0, i] = m, 1.0
        score = float(self.model(torch.from_numpy(X).to(self.device), torch.from_numpy(ok).to(self.device)))
        cry = score >= self.threshold
        self.cry_run = self.cry_run + 1 if cry else 0
        return {"score": score, "cry": cry, "alert": self.cry_run == self.alert_s}   # one alert per cry episode

# 2. Deployment file

Writes `CRNN_deploy.json` next to `CRNN.pt`: the decision threshold (best F1 on the out-of-fold validation scores), the
alert rule and the log-mel settings and statistics saved with the model.

In [ ]:
# ===== Deployment file of the CRNN: threshold + settings (read-only on the training results, writes one small JSON) =====
try:
    from google.colab import drive; drive.mount("/content/drive")
except ImportError:
    pass
import os, sys, json, importlib
sys.path.insert(0, os.path.join(os.environ.get("MAHD_ROOT", "/content/drive/MyDrive/Mahd"), "code"))
import mahd_test_common as T; importlib.reload(T)
import torch
np, pd = T.np, T.pd

oof = pd.read_parquet(T.OOF_PATH); oof = oof[oof["fold"].ge(0)]                  # out-of-fold validation scores
s, yo = oof[f"score | {T.NAMES['CRNN']}"].to_numpy(), oof["y"].to_numpy()
p, r, t = T.precision_recall_curve(yo, s); f = 2 * p * r / np.maximum(p + r, 1e-12); i = int(np.argmax(f[:-1]))
ck = torch.load(os.path.join(T.FINAL, "CRNN.pt"), map_location="cpu")
deploy = {"model": "CRNN.pt", "threshold": float(t[i]), "alert_seconds": 2, "context_s": ck["context_s"],
          "logmel": ck["logmel"], "logmel_stats": list(ck["logmel_stats"]), "window_norm": ck["window_norm"],
          "validation_at_threshold": {"precision": float(p[i]), "recall": float(r[i]), "F1": float(f[i])}}
json.dump(deploy, open(os.path.join(T.FINAL, "CRNN_deploy.json"), "w"), indent=1)
print(json.dumps(deploy, indent=1))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{
 "model": "CRNN.pt",
 "threshold": 2.900390625,
 "alert_seconds": 2,
 "context_s": 4,
 "logmel": {
  "sr": 16000,
  "n_fft": 1024,
  "win_length": 400,
  "hop_length": 160,
  "n_mels": 64,
  "fmin": 50,
  "fmax": 8000,
  "center": false,
  "db": "10*log10(max(power, 1e-10))"
 },
 "logmel_stats": [
  -31.47265625,
  15.005508422851562
 ],
 "window_norm": {
  "target_db_rms": -20.0,
  "floor_db": -50.0
 },
 "validation_at_threshold": {
  "precision": 0.8965293545248135,
  "recall": 0.8456293706293706,
  "F1": 0.8703357999145318
 }
}


# 3. Example

Runs the detector second by second over one recording.

In [ ]:
# ===== Example: run the detector on one recording, second by second =====
import soundfile as sf
FINAL_DIR = "/content/drive/MyDrive/Mahd/models/final_detection"
det = CryDetector(os.path.join(FINAL_DIR, "CRNN.pt"), os.path.join(FINAL_DIR, "CRNN_deploy.json"))

audio, sr = sf.read("example.wav", always_2d=True, dtype="float32")          # any recording
audio = librosa.resample(audio.mean(axis=1), orig_sr=sr, target_sr=SR) if sr != SR else audio.mean(axis=1)
for k in range(len(audio) // WIN):
    out = det.step(audio[k * WIN:(k + 1) * WIN])
    print(f"{k:4d} s  score {out['score']:+.2f}  {'CRY' if out['cry'] else '   '}  {'<- ALERT' if out['alert'] else ''}")

# Appendix — Final training and test evaluation

The cells that trained the final CRNN on all train/validation windows and evaluated it once on the test set (saved
outputs). The shared-code cell writes `mahd_test_common.py`; the training cell also trains the 1-s CNN, which is used in
the result table to show the effect of the temporal context; the baseline row (gradient boosting on 215 hand-crafted
features) was trained with the same protocol. Then the ensemble check, the bootstrap confidence intervals, Macro-F1 and
the CPU speed.

In [ ]:
# ===== T0: shared code for T1-T4 (run once; writes Mahd/code/mahd_test_common.py) =====
try:
    from google.colab import drive; drive.mount("/content/drive")
except ImportError:
    pass
import os
MAHD_ROOT = os.environ.get("MAHD_ROOT", "/content/drive/MyDrive/Mahd")
COMMON = r'''
import os, sys, glob, json, time, gc
import numpy as np, pandas as pd
from sklearn.metrics import average_precision_score, precision_recall_curve, matthews_corrcoef
MAHD_ROOT = os.environ.get("MAHD_ROOT", "/content/drive/MyDrive/Mahd")
sys.path.insert(0, os.path.join(MAHD_ROOT, "code")); import mahd_features as MF
MODELS = os.path.join(MAHD_ROOT, "models"); FINAL = os.path.join(MODELS, "final_detection"); os.makedirs(FINAL, exist_ok=True)
OOF_PATH = os.path.join(MODELS, "oof_fullband_experiments.parquet")
TEST_PATH = os.path.join(FINAL, "test_scores.parquet")
EVENTS = {k: [tuple(e) for e in v] for k, v in json.load(open(os.path.join(MAHD_ROOT, "icsd_cry_events.json"))).items()}
NORM_TARGET_DB, NORM_FLOOR_DB = -20.0, -50.0
META = ["audio_path", "start_s", "split", "win_s", "fold", "dataset", "subset", "original_label", "binary_label", "level_db", "level_db_bl"]
FEAT = MF.FEATURE_NAMES; BL = ["bl_" + n for n in FEAT]
NAMES = {"B2": "B2 YAMNet zero-shot", "B3": "B3 LR | 215 | no context", "B4": "B4 SVM | 215 | no context",
         "B5": "B5 HGB | 215 | no context", "CNN1": "CNN 1 s | log-mel | full band + aug | norm=window",
         "CNN4": "CNN 4 s | log-mel | full band + aug | norm=window", "CRNN": "CRNN 4 s | log-mel | full band + aug | norm=window"}

def load_split(folder, cols, split, npys=()):
    """Windows of one split (exactly 1 s), sorted by recording and time; npys = suffixes of .npy files to load."""
    metas, arrs = [], {s: [] for s in npys}
    for p in sorted(glob.glob(os.path.join(MAHD_ROOT, folder, "chunk_*.parquet"))):
        m = pd.read_parquet(p, columns=cols)
        keep = (m["split"].eq(split) & (m["win_s"] >= 1 - 1e-9)).to_numpy()
        metas.append(m[keep])
        for s in npys:
            arrs[s].append(np.load(p[:-len(".parquet")] + s + ".npy", mmap_mode="r")[keep])
    m = pd.concat(metas, ignore_index=True)
    o = np.lexsort((m["start_s"].to_numpy(), m["audio_path"].to_numpy()))
    return m.iloc[o].reset_index(drop=True), {s: np.concatenate(a)[o] for s, a in arrs.items()}

def same_windows(a, b, name):
    assert len(a) == len(b) and (a["audio_path"].to_numpy() == b["audio_path"].to_numpy()).all() \
        and np.allclose(a["start_s"], b["start_s"]), f"{name}: windows differ"

def add_labels(d):
    d["y"] = d["binary_label"].eq("Cry").astype(int)
    bg_ = d["subset"].eq("Materials Background"); st_ = d["dataset"].eq("ICSD") & ~bg_
    d["source"] = np.select([bg_, st_ & d["original_label"].eq("Infantcry") & d["y"].eq(0), st_, d["dataset"].eq("Nonspeech7k")],
                            ["BG: " + d["original_label"], "ICSD outside event", "ICSD: " + d["original_label"],
                             "NS7k: " + d["original_label"]], default=d["dataset"])
    amb = np.zeros(len(d), bool); icsd_ = (d["dataset"].eq("ICSD") & d["subset"].ne("Materials Background")).to_numpy()
    for i in np.flatnonzero(icsd_):
        evs = EVENTS.get(os.path.basename(d["audio_path"].iat[i]), []); a = d["start_s"].iat[i]; b = a + 1.0
        ov = sum(max(0.0, min(b, off) - max(a, on)) for on, off in evs)
        dist = min((max(0.0, on - b, a - off) for on, off in evs), default=np.inf)
        amb[i] = (ov < 0.25) if d["y"].iat[i] == 1 else (dist < 0.5)
    d["ambiguous"] = amb
    return d

def prev_index(d, lags=4):
    """P[i] = [i, the second 1 s before, ..., lags s before] in the same recording, -1 if there is none."""
    key = pd.DataFrame({"audio_path": d["audio_path"], "t": np.round(d["start_s"], 3), "pos": np.arange(len(d))})
    P = np.full((len(d), lags + 1), -1, np.int64); P[:, 0] = np.arange(len(d))
    for lag in range(1, lags + 1):
        pr = key.merge(key.assign(t=np.round(key["t"] + lag, 3)), on=["audio_path", "t"], suffixes=("", "_prev"))
        P[pr["pos"].to_numpy(), lag] = pr["pos_prev"].to_numpy()
    return P

def weights(y, src):
    w = np.ones(len(y)); non = y == 0; s = pd.Series(src)
    w[non] = non.sum() / s[non].nunique() / s[non].map(s[non].value_counts()).to_numpy()
    w[~non] = non.sum() / (~non).sum()
    return w / w.mean()

def norm_gain(lev):
    """Per-window loudness normalisation as a dB gain (exact shift of a log-mel)."""
    return (NORM_TARGET_DB - np.maximum(lev, NORM_FLOOR_DB)).astype(np.float32)

def test_keys():
    te, _ = load_split("features_detect_D_win", META, "test")
    return add_labels(te)

def saved_test(name):
    if os.path.exists(TEST_PATH):
        o = pd.read_parquet(TEST_PATH)
        if f"score | {name}" in o.columns:
            return o[f"score | {name}"].to_numpy(), o[f"score_bl | {name}"].to_numpy()
    return None

def save_test(name, s, s_bl, te):
    o = pd.read_parquet(TEST_PATH) if os.path.exists(TEST_PATH) else te[["audio_path", "start_s", "dataset", "subset", "y"]].copy()
    assert len(o) == len(te) and (o["audio_path"].to_numpy() == te["audio_path"].to_numpy()).all(), "test windows differ"
    o[f"score | {name}"] = s; o[f"score_bl | {name}"] = s_bl
    o.to_parquet(TEST_PATH, index=False)

# ---------- neural networks (same as the experiments) ----------
def nn_classes():
    import torch, torch.nn as nn, torch.nn.functional as F
    class Encoder(nn.Module):
        def __init__(self, ch=(16, 32, 64, 128), emb=128):
            super().__init__(); layers, c_in = [], 1
            for c in ch:
                layers += [nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                           nn.Conv2d(c, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True), nn.AvgPool2d(2)]
                c_in = c
            self.conv = nn.Sequential(*layers); self.fc = nn.Linear(2 * c_in, emb)
        def forward(self, x):
            h = self.conv(x.unsqueeze(1)).mean(2)
            return F.relu(self.fc(torch.cat([h.mean(-1), h.amax(-1)], 1)))
    class CNN1(nn.Module):
        def __init__(self):
            super().__init__(); self.enc = Encoder(); self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(128, 1))
        def forward(self, x, ok):
            return self.head(self.enc(x[:, 0]))
    class CNNStack(nn.Module):
        def __init__(self, n=5):
            super().__init__(); self.enc = Encoder()
            self.head = nn.Sequential(nn.Linear(n * 128 + n - 1, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 1))
        def forward(self, x, ok):
            b, n = x.shape[:2]
            e = self.enc(x.flatten(0, 1)).view(b, n, -1) * ok[..., None]
            return self.head(torch.cat([e.flatten(1), ok[:, 1:]], 1))
    class CRNN(nn.Module):
        def __init__(self):
            super().__init__(); self.enc = Encoder(); self.gru = nn.GRU(129, 128, batch_first=True)
            self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(128, 1))
        def forward(self, x, ok):
            b, n = x.shape[:2]
            e = self.enc(x.flatten(0, 1)).view(b, n, -1) * ok[..., None]
            out, _ = self.gru(torch.cat([e, ok[..., None]], 2).flip(1).float())
            return self.head(out[:, -1])
    return {"CNN1": (CNN1, 0), "CNN4": (CNNStack, 4), "CRNN": (CRNN, 4)}
'''
os.makedirs(os.path.join(MAHD_ROOT, "code"), exist_ok=True)
with open(os.path.join(MAHD_ROOT, "code", "mahd_test_common.py"), "w") as f:
    f.write(COMMON)
print("written:", os.path.join(MAHD_ROOT, "code", "mahd_test_common.py"))

Mounted at /content/drive
written: /content/drive/MyDrive/Mahd/code/mahd_test_common.py


In [ ]:
# ===== T2: CNN 1 s and CRNN 4 s — final training on all train/validation + test scores (GPU + large RAM, ~45-60 min) =====
try:
    from google.colab import drive; drive.mount("/content/drive")
except ImportError:
    pass
import os, sys, importlib
sys.path.insert(0, os.path.join(os.environ.get("MAHD_ROOT", "/content/drive/MyDrive/Mahd"), "code"))
import mahd_test_common as T; importlib.reload(T)
import torch, torch.nn.functional as F
np, pd = T.np, T.pd
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu"); AMP = DEV.type == "cuda"
print("Device:", torch.cuda.get_device_name(0) if AMP else "CPU (too slow for training: switch to a GPU)")

cols = ["audio_path", "start_s", "split", "win_s", "fold", "dataset", "subset", "original_label", "binary_label", "level_db", "level_db_bl"]
tv, mtv = T.load_split("features_detect_D_file_logmel", cols, "trainval", npys=("", "_bl")); tv = T.add_labels(tv)
te, mte = T.load_split("features_detect_D_file_logmel", cols, "test", npys=("", "_bl")); te = T.add_labels(te)
T.same_windows(te, T.test_keys(), "log-mel test vs 215-feature test")
P_tv, P_te = T.prev_index(tv), T.prev_index(te)
tr = np.flatnonzero(~tv["ambiguous"].to_numpy()); y = tv["y"].to_numpy(); w = T.weights(y[tr], tv["source"].to_numpy()[tr])
G = lambda d: (torch.tensor(T.norm_gain(d["level_db"].to_numpy()), device=DEV), torch.tensor(T.norm_gain(d["level_db_bl"].to_numpy()), device=DEV))
TV = {"D": torch.tensor(mtv[""], device=DEV), "A": torch.tensor(mtv["_bl"], device=DEV), "P": torch.tensor(P_tv, device=DEV), "G": G(tv)}
TE = {"D": torch.tensor(mte[""], device=DEV), "A": torch.tensor(mte["_bl"], device=DEV), "P": torch.tensor(P_te, device=DEV), "G": G(te)}
del mtv, mte; T.gc.collect()
sel = np.random.default_rng(0).choice(tr, min(len(tr), 20000), replace=False)
smp = TV["D"][torch.tensor(sel, device=DEV)].float() + TV["G"][0][torch.tensor(sel, device=DEV)][:, None, None]
STATS = (float(smp.mean()), float(smp.std())); del smp
print(f"train: {len(tr):,} windows | test: {len(te):,} | log-mel stats (mean, std) = {STATS[0]:.2f}, {STATS[1]:.2f}")

def spec_augment(x):
    n, f, t = x.shape
    f0 = torch.randint(0, f, (n, 1, 1), device=x.device); fw = torch.randint(0, 9, (n, 1, 1), device=x.device)
    t0 = torch.randint(0, t, (n, 1, 1), device=x.device); tw = torch.randint(0, 13, (n, 1, 1), device=x.device)
    fr = torch.arange(f, device=x.device)[None, :, None]; tt = torch.arange(t, device=x.device)[None, None, :]
    return x * ~(((fr >= f0) & (fr < f0 + fw)) | ((tt >= t0) & (tt < t0 + tw)))

def batch(S, idx, lags, train, bl=False):
    p = S["P"][idx, :lags + 1]; ok = p >= 0; q = p.clamp(min=0)
    xd = S["D"][q].float() + S["G"][0][q][..., None, None]; xa = S["A"][q].float() + S["G"][1][q][..., None, None]
    x = torch.where((torch.rand(len(idx), device=DEV) < 0.5)[:, None, None, None], xa, xd) if train else (xa if bl else xd)
    x = (x - STATS[0]) / STATS[1]
    if train:
        x = spec_augment(x.flatten(0, 1)).view_as(x)
    return x * ok[..., None, None], ok.float()

def final_nn(key, epochs=15, lr=2e-3, bs=256):
    name = T.NAMES[key]; path = os.path.join(T.FINAL, key + ".pt")
    if T.saved_test(name) is not None and os.path.exists(path):
        print(f"{name}: already done"); return
    cls, lags = T.nn_classes()[key]; torch.manual_seed(42); np.random.seed(42); t0 = T.time.time()
    model = cls().to(DEV); opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    steps = len(tr) // bs; sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * steps, pct_start=0.2)
    scaler = torch.amp.GradScaler(enabled=AMP); tr_t = torch.tensor(tr, device=DEV)
    W = torch.zeros(len(tv), device=DEV); W[tr_t] = torch.tensor(w, dtype=torch.float32, device=DEV)
    Y = torch.tensor(y, dtype=torch.float32, device=DEV)
    for ep in range(epochs):
        model.train(); perm = tr_t[torch.randperm(len(tr_t), device=DEV)]
        for i in range(steps):
            idx = perm[i * bs:(i + 1) * bs]
            with torch.autocast(DEV.type, dtype=torch.float16, enabled=AMP):
                logit = model(*batch(TV, idx, lags, True)).squeeze(-1)
            loss = (F.binary_cross_entropy_with_logits(logit.float(), Y[idx], reduction="none") * W[idx]).mean()
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
    model.eval(); out = {}
    for bl in (False, True):
        parts = []
        with torch.no_grad(), torch.autocast(DEV.type, dtype=torch.float16, enabled=AMP):
            for i in range(0, len(te), 1024):
                idx = torch.arange(i, min(i + 1024, len(te)), device=DEV)
                parts.append(model(*batch(TE, idx, lags, False, bl)).squeeze(-1).float().cpu().numpy())
        out[bl] = np.concatenate(parts)
    torch.save({"state_dict": {k: v.cpu() for k, v in model.state_dict().items()}, "model": key, "context_s": lags,
                "logmel_stats": STATS, "logmel": {"sr": 16000, "n_fft": 1024, "win_length": 400, "hop_length": 160,
                "n_mels": 64, "fmin": 50, "fmax": 8000, "center": False, "db": "10*log10(max(power, 1e-10))"},
                "window_norm": {"target_db_rms": T.NORM_TARGET_DB, "floor_db": T.NORM_FLOOR_DB}}, path)
    T.save_test(name, out[False], out[True], te)
    print(f"{name}: trained in {T.time.time() - t0:.0f} s | {os.path.getsize(path) / 1e6:.2f} MB")
    del model, opt; T.gc.collect(); torch.cuda.empty_cache() if AMP else None

for key in ["CNN1", "CRNN"]:
    final_nn(key)

Mounted at /content/drive
Device: NVIDIA L4
train: 149,325 windows | test: 54,002 | log-mel stats (mean, std) = -31.47, 15.01
CNN 1 s | log-mel | full band + aug | norm=window: trained in 246 s | 1.32 MB
CRNN 4 s | log-mel | full band + aug | norm=window: trained in 1333 s | 1.72 MB


In [ ]:
# ===== T3: the comparison table on the TEST set (no training, ~2 min) =====
# Thresholds and ensemble constants come only from the out-of-fold validation scores (no tuning on the test set).
try:
    from google.colab import drive; drive.mount("/content/drive")
except ImportError:
    pass
import os, sys, importlib
sys.path.insert(0, os.path.join(os.environ.get("MAHD_ROOT", "/content/drive/MyDrive/Mahd"), "code"))
import mahd_test_common as T; importlib.reload(T)
from IPython.display import display
np, pd = T.np, T.pd
te = T.test_keys(); ts = pd.read_parquet(T.TEST_PATH); T.same_windows(te, ts, "test scores")
P = T.prev_index(te, lags=1); yte = te["y"].to_numpy(); don = te["dataset"].eq("Donate-a-Cry").to_numpy()
oof = pd.read_parquet(T.OOF_PATH); oof = oof[oof["fold"].ge(0)]

def best_f1_threshold(y, s):
    p, r, t = T.precision_recall_curve(y, s); f = 2 * p * r / np.maximum(p + r, 1e-12)
    return float(t[np.argmax(f[:-1])])

def device_metrics(pred):
    """Alert = 2 consecutive seconds called cry (same recording). Cry episode = consecutive cry seconds (>= 2 s)."""
    p1 = P[:, 1]; has = p1 >= 0
    on = pred & has & pred[np.where(has, p1, 0)]
    on_prev = has & on[np.where(has, p1, 0)]
    onset = on & ~on_prev
    false_onset = onset & (yte == 0) & ~(has & (yte[np.where(has, p1, 0)] == 1))
    start = (yte == 1) & ~(has & (yte[np.where(has, p1, 0)] == 1)); ep = np.cumsum(start) * (yte == 1)
    ep_df = pd.DataFrame({"ep": ep, "on": on, "pos": np.arange(len(yte))})[ep > 0]
    g = ep_df.groupby("ep"); length = g.size(); det = g["on"].any()
    first_on = ep_df[ep_df["on"]].groupby("ep")["pos"].min(); first = g["pos"].min()
    keep = length >= 2
    delay = (first_on - first.reindex(first_on.index)).reindex(det[keep & det].index)
    return {"cry episodes detected": float(det[keep].mean()), "alert delay (s, mean)": float(delay.mean() + 1),
            "false alerts per hour": float(false_onset.sum() / max((yte == 0).sum() / 3600, 1e-9))}

def evaluate(s, s_bl, thr):
    pred = s >= thr; tp = (pred & (yte == 1)).sum(); fp = (pred & (yte == 0)).sum(); fn = (~pred & (yte == 1)).sum()
    prec, rec = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    out = {"AP": T.average_precision_score(yte, s), "AP without Donate": T.average_precision_score(yte[~don], s[~don]),
           "AP band-limited input": T.average_precision_score(yte, s_bl),
           "precision": prec, "recall": rec, "F1": 2 * prec * rec / max(prec + rec, 1e-12),
           "MCC": T.matthews_corrcoef(yte, pred), "non-cry seconds flagged": fp / max((yte == 0).sum(), 1)}
    out.update(device_metrics(pred))
    return out

size = lambda f: os.path.getsize(os.path.join(T.FINAL, f)) / 1e6 if os.path.exists(os.path.join(T.FINAL, f)) else np.nan
rows, Z = {}, {}
rows["B0 chance"] = {"AP": yte.mean(), "AP without Donate": yte[~don].mean()}
for key, label, f in [("B5", "Baseline: gradient boosting, 215 features, 1 s", "B5.joblib"),
                      ("CNN1", "CNN from scratch, log-mel, 1 s", "CNN1.pt"), ("CRNN", "CRNN, log-mel, 1 s + 4 s context", "CRNN.pt")]:
    n = T.NAMES[key]
    if f"score | {n}" not in ts.columns:
        print("not trained yet:", label); continue
    so = oof[f"score | {n}"].to_numpy()
    rows[label] = {**evaluate(ts[f"score | {n}"].to_numpy(), ts[f"score_bl | {n}"].to_numpy(), best_f1_threshold(oof["y"].to_numpy(), so)),
                   "size (MB)": size(f)}
    Z[key] = (np.mean(so), np.std(so))
table = pd.DataFrame(rows).T
display(table.round(3))
table.to_csv(os.path.join(T.FINAL, "test_results.csv"))
print(f"test: {len(te):,} seconds, {te['audio_path'].nunique():,} recordings, Cry {yte.mean():.1%} | saved", os.path.join(T.FINAL, "test_results.csv"))
print("Threshold of each model = best F1 on its out-of-fold validation scores. Alert = 2 consecutive cry seconds.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,AP,AP without Donate,AP band-limited input,precision,recall,F1,MCC,non-cry seconds flagged,cry episodes detected,"alert delay (s, mean)",false alerts per hour,size (MB)
B0 chance,0.059,0.049,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"Baseline: gradient boosting, 215 features, 1 s",0.824,0.799,0.814,0.678,0.793,0.731,0.715,0.024,0.769,2.212,13.461,0.917
"CNN from scratch, log-mel, 1 s",0.898,0.879,0.888,0.841,0.834,0.837,0.827,0.010,0.811,2.187,4.676,1.324
"CRNN, log-mel, 1 s + 4 s context",0.940,0.928,0.935,0.905,0.867,0.885,0.879,0.006,0.846,2.196,2.976,1.723


test: 54,002 seconds, 5,932 recordings, Cry 5.9% | saved /content/drive/MyDrive/Mahd/models/final_detection/test_results.csv
Threshold of each model = best F1 on its out-of-fold validation scores. Alert = 2 consecutive cry seconds.


In [ ]:
# ===== T3b: ensemble CRNN + baseline (equal-weight soft voting) — run right after T3, in the same session =====
# The ensemble's constants and threshold come from the validation (out-of-fold) scores, like every other model.
if {"B5", "CRNN"} <= set(Z):
    zt = lambda k, c: (ts[f"{c} | {T.NAMES[k]}"].to_numpy() - Z[k][0]) / Z[k][1]
    zo = lambda k: (oof[f"score | {T.NAMES[k]}"].to_numpy() - Z[k][0]) / Z[k][1]
    so = (zo("B5") + zo("CRNN")) / 2
    crnn_oof = oof["score | " + T.NAMES["CRNN"]].to_numpy()
    print(f"validation AP: ensemble {T.average_precision_score(oof['y'], so):.3f} | CRNN alone {T.average_precision_score(oof['y'], crnn_oof):.3f}")
    rows["Ensemble: CRNN + gradient boosting"] = {
        **evaluate((zt("B5", "score") + zt("CRNN", "score")) / 2, (zt("B5", "score_bl") + zt("CRNN", "score_bl")) / 2,
                   best_f1_threshold(oof["y"].to_numpy(), so)),
        "size (MB)": size("B5.joblib") + size("CRNN.pt")}
    table = pd.DataFrame(rows).T
    display(table.round(3))
    table.to_csv(os.path.join(T.FINAL, "test_results.csv"))
else:
    print("run T1, T2 and T3 first")

validation AP: ensemble 0.927 | CRNN alone 0.931


,AP,AP without Donate,AP band-limited input,precision,recall,F1,MCC,non-cry seconds flagged,cry episodes detected,"alert delay (s, mean)",false alerts per hour,size (MB)
B0 chance,0.059,0.049,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"Baseline: gradient boosting, 215 features, 1 s",0.824,0.799,0.814,0.678,0.793,0.731,0.715,0.024,0.769,2.212,13.461,0.917
"CNN from scratch, log-mel, 1 s",0.898,0.879,0.888,0.841,0.834,0.837,0.827,0.010,0.811,2.187,4.676,1.324
"CRNN, log-mel, 1 s + 4 s context",0.940,0.928,0.935,0.905,0.867,0.885,0.879,0.006,0.846,2.196,2.976,1.723
Ensemble: CRNN + gradient boosting,0.936,0.921,0.930,0.876,0.881,0.879,0.871,0.008,0.843,2.125,3.967,2.640


In [ ]:
# ===== T3c: are the test differences real? Bootstrap over test recordings (no training, ~3 min; run after T3/T3b) =====
# Rule fixed BEFORE looking: the model with the highest test AP is chosen; if its 95% interval of the AP difference
# with a simpler model includes 0, the two are tied and the simpler/lighter one is chosen.
S = {}
for key, label in [("B5", "Baseline: gradient boosting"), ("CNN1", "CNN 1 s"), ("CRNN", "CRNN 4 s")]:
    if f"score | {T.NAMES[key]}" in ts.columns:
        S[label] = ts[f"score | {T.NAMES[key]}"].to_numpy()
if "Ensemble: CRNN + gradient boosting" in rows and {"B5", "CRNN"} <= set(Z):
    S["Ensemble: CRNN + gradient boosting"] = (zt("B5", "score") + zt("CRNN", "score")) / 2
groups = list(te.groupby("audio_path").indices.values()); rng = np.random.default_rng(0)
boot = {k: [] for k in S}
for _ in range(300):
    idx = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
    for k, s in S.items():
        boot[k].append(T.average_precision_score(yte[idx], s[idx]))
boot = {k: np.array(v) for k, v in boot.items()}
ci = pd.DataFrame({k: {"test AP": T.average_precision_score(yte, S[k]), "95% low": np.percentile(v, 2.5),
                       "95% high": np.percentile(v, 97.5)} for k, v in boot.items()}).T
display(ci.round(3))
best = ci["test AP"].idxmax()
diff = pd.DataFrame({k: {"AP difference (best - this)": float(np.mean(boot[best] - v)),
                         "95% low": np.percentile(boot[best] - v, 2.5), "95% high": np.percentile(boot[best] - v, 97.5),
                         "real difference?": "yes" if np.percentile(boot[best] - v, 2.5) > 0 else "no (tie)"}
                     for k, v in boot.items() if k != best}).T
print(f"highest test AP: {best}")
display(diff)
print("Bootstrap over test recordings, 300 resamples. 'no (tie)' = the interval includes 0.")

,test AP,95% low,95% high
Baseline: gradient boosting,0.824,0.808,0.839
CNN 1 s,0.898,0.884,0.908
CRNN 4 s,0.940,0.930,0.949
Ensemble: CRNN + gradient boosting,0.936,0.925,0.945


highest test AP: CRNN 4 s


,AP difference (best - this),95% low,95% high,real difference?
Baseline: gradient boosting,0.116211,0.104212,0.129473,yes
CNN 1 s,0.043156,0.035505,0.050956,yes
Ensemble: CRNN + gradient boosting,0.004885,0.000176,0.009421,yes


Bootstrap over test recordings, 300 resamples. 'no (tie)' = the interval includes 0.


In [ ]:
# ===== T3d (standalone): macro F1 on the test set, same thresholds as T3 (from validation) — no training, ~1 min =====
try:
    from google.colab import drive; drive.mount("/content/drive")
except ImportError:
    pass
import os, sys, importlib
sys.path.insert(0, os.path.join(os.environ.get("MAHD_ROOT", "/content/drive/MyDrive/Mahd"), "code"))
import mahd_test_common as T; importlib.reload(T)
from sklearn.metrics import f1_score
from IPython.display import display
np, pd = T.np, T.pd
te = T.test_keys(); ts = pd.read_parquet(T.TEST_PATH); T.same_windows(te, ts, "test scores"); yte = te["y"].to_numpy()
oof = pd.read_parquet(T.OOF_PATH); oof = oof[oof["fold"].ge(0)]; yo = oof["y"].to_numpy()

def best_f1_threshold(y, s):
    p, r, t = T.precision_recall_curve(y, s); f = 2 * p * r / np.maximum(p + r, 1e-12)
    return float(t[np.argmax(f[:-1])])

def f1s(pred):
    return {"F1 (cry)": f1_score(yte, pred), "F1 (non-cry)": f1_score(1 - yte, ~pred), "macro F1": f1_score(yte, pred, average="macro")}

mf, Z = {}, {}
for key, label in [("B5", "Baseline: gradient boosting, 215 features, 1 s"), ("CNN1", "CNN from scratch, log-mel, 1 s"),
                   ("CRNN", "CRNN, log-mel, 1 s + 4 s context")]:
    n = T.NAMES[key]; so = oof[f"score | {n}"].to_numpy(); Z[key] = (np.mean(so), np.std(so))
    mf[label] = f1s(ts[f"score | {n}"].to_numpy() >= best_f1_threshold(yo, so))
z = lambda k, s: (s - Z[k][0]) / Z[k][1]
ens_oof = (z("B5", oof[f"score | {T.NAMES['B5']}"].to_numpy()) + z("CRNN", oof[f"score | {T.NAMES['CRNN']}"].to_numpy())) / 2
ens_te = (z("B5", ts[f"score | {T.NAMES['B5']}"].to_numpy()) + z("CRNN", ts[f"score | {T.NAMES['CRNN']}"].to_numpy())) / 2
mf["Ensemble: CRNN + gradient boosting"] = f1s(ens_te >= best_f1_threshold(yo, ens_oof))
display(pd.DataFrame(mf).T.round(3))

Mounted at /content/drive


,F1 (cry),F1 (non-cry),macro F1
"Baseline: gradient boosting, 215 features, 1 s",0.731,0.982,0.856
"CNN from scratch, log-mel, 1 s",0.837,0.990,0.914
"CRNN, log-mel, 1 s + 4 s context",0.885,0.993,0.939
Ensemble: CRNN + gradient boosting,0.879,0.992,0.935


In [ ]:
# ===== T4 (optional): speed per 1 s of audio on this CPU (one thread), features + model =====
try:
    from google.colab import drive; drive.mount("/content/drive")
except ImportError:
    pass
import os, sys, importlib, joblib, time, librosa
sys.path.insert(0, os.path.join(os.environ.get("MAHD_ROOT", "/content/drive/MyDrive/Mahd"), "code"))
import mahd_test_common as T; importlib.reload(T)
import torch; torch.set_num_threads(1)
np = T.np
w = (np.random.default_rng(0).normal(size=16000) * 0.05).astype(np.float32)
def ms(f, n=30):
    f(); t0 = time.perf_counter()
    for _ in range(n): f()
    return (time.perf_counter() - t0) / n * 1000
feat = np.array([list(T.MF.window_features(w, "D").values())], np.float32)
mel_cfg = dict(sr=16000, n_fft=1024, win_length=400, hop_length=160, n_mels=64, fmin=50, fmax=8000, center=False)
rows = {"215 features (per second)": ms(lambda: T.MF.window_features(w, "D"), 10),
        "log-mel (per second)": ms(lambda: librosa.feature.melspectrogram(y=w, **mel_cfg))}
for key in ["B5"]:
    p = os.path.join(T.FINAL, key + ".joblib")
    if os.path.exists(p):
        m = joblib.load(p)["model"]; rows[f"{key} model"] = ms(lambda: m.decision_function(np.nan_to_num(feat)))
for key in ["CNN1", "CRNN"]:
    p = os.path.join(T.FINAL, key + ".pt")
    if os.path.exists(p):
        ck = torch.load(p, map_location="cpu"); cls, lags = T.nn_classes()[key]; m = cls(); m.load_state_dict(ck["state_dict"]); m.eval()
        x = torch.zeros(1, lags + 1, 64, 94); ok = torch.ones(1, lags + 1)
        with torch.no_grad():
            rows[f"{key} model (all {lags + 1} s recomputed)"] = ms(lambda: m(x, ok))
print("milliseconds per 1 s of audio (a Raspberry Pi is several times slower):")
for k, v in rows.items():
    print(f"  {k:45s} {v:8.2f} ms")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
milliseconds per 1 s of audio (a Raspberry Pi is several times slower):
  215 features (per second)                        19.39 ms
  log-mel (per second)                              2.36 ms
  B5 model                                          4.84 ms
  CNN1 model (all 1 s recomputed)                   4.47 ms
  CRNN model (all 5 s recomputed)                  16.16 ms


In [ ]:
# ===== Export Mahd Cry Detection CRNN (+ librosa-exact Log-Mel) to ONNX =====
!pip install -q onnx onnxruntime

import os, json, inspect
import numpy as np
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import onnxruntime as ort

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

FINAL_DIR = "/content/drive/MyDrive/Mahd/models/final_detection"
PT_PATH = os.path.join(FINAL_DIR, "CRNN.pt")
DEPLOY_PATH = os.path.join(FINAL_DIR, "CRNN_deploy.json")
ONNX_PATH = os.path.join(FINAL_DIR, "mahd_cry_detection_crnn.onnx")
META_PATH = os.path.join(FINAL_DIR, "mahd_cry_detection_meta.json")

SR = 16000
WIN = 16000
CONTEXT_S = 4
N_FFT = 1024
WIN_LENGTH = 400
HOP_LENGTH = 160
N_MELS = 64
FMIN, FMAX = 50, 8000
NORM_TARGET_DB, NORM_FLOOR_DB = -20.0, -50.0


# 1) The CRNN (identical to the trained model)
class Encoder(nn.Module):
    def __init__(self, ch=(16, 32, 64, 128), emb=128):
        super().__init__()
        layers, c_in = [], 1
        for c in ch:
            layers += [
                nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                nn.Conv2d(c, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                nn.AvgPool2d(2),
            ]
            c_in = c
        self.conv = nn.Sequential(*layers)
        self.fc = nn.Linear(2 * c_in, emb)

    def forward(self, x):
        h = self.conv(x.unsqueeze(1)).mean(2)
        return F.relu(self.fc(torch.cat([h.mean(-1), h.amax(-1)], 1)))


class CRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = Encoder()
        self.gru = nn.GRU(129, 128, batch_first=True)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(128, 1))

    def forward(self, x, ok):
        b, n = x.shape[:2]
        e = self.enc(x.flatten(0, 1)).view(b, n, -1) * ok[..., None]
        out, _ = self.gru(torch.cat([e, ok[..., None]], 2).flip(1).float())
        return self.head(out[:, -1])


# 2) Wrapper that puts STFT + Mel + loudness normalisation + standardisation inside the ONNX graph
class CRNNEndToEndONNX(nn.Module):
    def __init__(self, crnn: CRNN, mean: float, std: float):
        super().__init__()
        self.crnn = crnn

        # DFT kernels identical to librosa.stft(n_fft=1024, win_length=400, hop_length=160, center=False)
        hann = librosa.filters.get_window("hann", WIN_LENGTH, fftbins=True)
        pad_left = (N_FFT - WIN_LENGTH) // 2  # 312: librosa centres the 400-sample window inside the 1024-point frame
        n = np.arange(pad_left, pad_left + WIN_LENGTH, dtype=np.float64)
        k = np.arange(0, N_FFT // 2 + 1, dtype=np.float64)[:, None]
        angles = 2.0 * np.pi * k * n[None, :] / N_FFT

        real_kernel = (hann[None, :] * np.cos(angles)).astype(np.float32)[:, None, :]
        imag_kernel = (-hann[None, :] * np.sin(angles)).astype(np.float32)[:, None, :]
        mel_basis = librosa.filters.mel(
            sr=SR, n_fft=N_FFT, n_mels=N_MELS, fmin=FMIN, fmax=FMAX
        ).astype(np.float32)

        self.pad_left = pad_left
        self.register_buffer("real_kernel", torch.from_numpy(real_kernel))
        self.register_buffer("imag_kernel", torch.from_numpy(imag_kernel))
        self.register_buffer("mel_basis", torch.from_numpy(mel_basis))
        self.register_buffer("mean", torch.tensor(float(mean), dtype=torch.float32))
        self.register_buffer("std", torch.tensor(float(std), dtype=torch.float32))

    def forward(self, audio, ok):
        # audio: (B, 5, 16000) raw float32 PCM in [-1, 1] (no peak normalisation); index 0 = the current second
        # ok:    (B, 5) float32 (1.0 = the second is available, 0.0 = missing)
        b, n_sec, samples = audio.shape
        w = audio.reshape(b * n_sec, samples)

        # Loudness normalisation (same as loudness_gain_db)
        rms = torch.sqrt(torch.clamp(torch.mean(w * w, dim=-1), min=0.0)) + 1e-10
        level_db = 20.0 * torch.log10(rms)
        gain_db = NORM_TARGET_DB - torch.clamp(level_db, min=NORM_FLOOR_DB)

        # Log-mel (same as librosa.feature.melspectrogram(..., center=False, power=2.0))
        w_slice = w[:, self.pad_left : samples - self.pad_left].unsqueeze(1)
        real = F.conv1d(w_slice, self.real_kernel, stride=HOP_LENGTH)
        imag = F.conv1d(w_slice, self.imag_kernel, stride=HOP_LENGTH)
        power = real * real + imag * imag
        mel = torch.matmul(self.mel_basis, power)
        logmel = 10.0 * torch.log10(torch.clamp(mel, min=1e-10))

        # Standardisation with the training statistics
        x = (logmel + gain_db[:, None, None] - self.mean) / self.std
        x = x.reshape(b, n_sec, N_MELS, -1)  # (B, 5, 64, 94)

        return self.crnn(x, ok)  # (B, 1) cry score (logit)


# 3) Load the trained weights and the deployment settings
ck = torch.load(PT_PATH, map_location="cpu", weights_only=False)
cfg = json.load(open(DEPLOY_PATH, "r", encoding="utf-8"))
mean, std = ck["logmel_stats"]

crnn = CRNN()
crnn.load_state_dict(ck["state_dict"])
crnn.eval()

wrapper = CRNNEndToEndONNX(crnn, mean=mean, std=std).eval()

# 4) Export to ONNX (one self-contained file, usable with onnxruntime in Flutter)
dummy_audio = torch.zeros(1, CONTEXT_S + 1, WIN, dtype=torch.float32)
dummy_ok = torch.ones(1, CONTEXT_S + 1, dtype=torch.float32)

export_kwargs = dict(
    input_names=["audio", "ok"],
    output_names=["score"],
    opset_version=17,
    do_constant_folding=True,
)
if "dynamo" in inspect.signature(torch.onnx.export).parameters:
    export_kwargs["dynamo"] = False

torch.onnx.export(wrapper, (dummy_audio, dummy_ok), ONNX_PATH, **export_kwargs)

# 5) Save the metadata file for the Flutter app
meta = {
    "model": "mahd_cry_detection_crnn.onnx",
    "sample_rate": SR,
    "window_samples": WIN,
    "context_seconds": CONTEXT_S,
    "total_seconds": CONTEXT_S + 1,
    "input_audio_name": "audio",
    "input_ok_name": "ok",
    "output_score_name": "score",
    "threshold": float(cfg["threshold"]),
    "alert_seconds": int(cfg["alert_seconds"]),
    "note": "Input 'audio' is raw float32 mono PCM [1, 5, 16000] in [-1, 1] (index 0 = current second) WITHOUT peak normalization. Log-Mel, loudness norm, and standardization are built inside the ONNX graph.",
}
with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

# 6) Automatic check: ONNX Runtime must give the same score as PyTorch + librosa
sess = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])
rng = np.random.default_rng(42)
mel_cfg = dict(sr=SR, n_fft=N_FFT, win_length=WIN_LENGTH, hop_length=HOP_LENGTH, n_mels=N_MELS, fmin=FMIN, fmax=FMAX, center=False, power=2.0)

max_diff = 0.0
for valid_s in [1, 2, 5]:                      # 1, 2 and 5 available seconds (start of a stream and full context)
    aud = np.zeros((1, CONTEXT_S + 1, WIN), dtype=np.float32)
    ok = np.zeros((1, CONTEXT_S + 1), dtype=np.float32)
    X_ref = np.zeros((1, CONTEXT_S + 1, N_MELS, 94), dtype=np.float32)

    for i in range(valid_s):
        w = (rng.normal(size=WIN) * 0.05).astype(np.float32)
        aud[0, i] = w
        ok[0, i] = 1.0
        m = librosa.feature.melspectrogram(y=w, **mel_cfg)
        lm = 10.0 * np.log10(np.maximum(m, 1e-10))
        lv = 20.0 * np.log10(np.sqrt(np.mean(w.astype(np.float64) ** 2)) + 1e-10)
        gain = NORM_TARGET_DB - max(lv, NORM_FLOOR_DB)
        X_ref[0, i] = ((lm + gain - mean) / std).astype(np.float32)

    with torch.no_grad():
        pt_score = float(crnn(torch.from_numpy(X_ref), torch.from_numpy(ok)).item())
    onnx_score = float(sess.run(["score"], {"audio": aud, "ok": ok})[0][0, 0])
    diff = abs(pt_score - onnx_score)
    max_diff = max(max_diff, diff)
    print(f"valid_s={valid_s} | PyTorch={pt_score:+.6f} | ONNX={onnx_score:+.6f} | diff={diff:.2e}")

size_mb = os.path.getsize(ONNX_PATH) / (1024 * 1024)
print(f"\n✅ Exported: {ONNX_PATH} ({size_mb:.2f} MB)")
print(f"✅ Meta saved: {META_PATH}")
print(f"✅ Max abs diff vs PyTorch+librosa: {max_diff:.2e}")
assert max_diff < 1e-3, f"Mismatch too large: {max_diff}"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_353/2505827747.py:145: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(wrapper, (dummy_audio, dummy_ok), ONNX_PATH, **export_kwargs)
/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:4604: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with GRU can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  return _generic_rnn(


valid_s=1 | PyTorch=-6.062794 | ONNX=-6.062793 | diff=9.54e-07
valid_s=2 | PyTorch=-7.042426 | ONNX=-7.042426 | diff=4.77e-07
valid_s=5 | PyTorch=-7.792356 | ONNX=-7.792355 | diff=1.43e-06

✅ Exported: /content/drive/MyDrive/Mahd/models/final_detection/mahd_cry_detection_crnn.onnx (3.33 MB)
✅ Meta saved: /content/drive/MyDrive/Mahd/models/final_detection/mahd_cry_detection_meta.json
✅ Max abs diff vs PyTorch+librosa: 1.43e-06
